<a href="https://colab.research.google.com/github/123321juliocesar/Desarrollo2/blob/main/REGLAS_DE_ASOCIACION_COMPLETO_CRISP_DM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de canasta de mercado mediante reglas de asociación

**Metodología:** CRISP-DM  
**Algoritmo principal:** Apriori  
**Algoritmo de contraste:** FP-Growth  
**Dataset:** `basket_analysis.csv`  
**Entorno recomendado:** Google Colab

---

## Arquitectura del análisis

```text
Google Drive / CSV
        ↓
Validación y comprensión de datos
        ↓
Preparación de matriz transaccional booleana
        ↓
Análisis exploratorio y coocurrencias
        ↓
Apriori: conjuntos frecuentes
        ↓
Reglas de asociación
        ↓
Evaluación: soporte, confianza, lift, leverage,
conviction, Jaccard y Kulczynski
        ↓
Recomendaciones comerciales y recomendador
        ↓
Exportación de tablas, gráficos y reporte
```

> Cada fila representa una transacción y cada columna representa un producto.  
> Las reglas identifican asociaciones estadísticas; no demuestran causalidad.

# 1. Comprensión del negocio

## 1.1 Problema

La empresa registra los productos incluidos en cada compra, pero todavía no conoce cuáles aparecen juntos con mayor frecuencia ni qué relaciones pueden aprovecharse en promociones, organización de productos y recomendaciones.

## 1.2 Objetivo general

Identificar patrones frecuentes de compra y generar reglas de asociación que permitan comprender qué productos tienden a adquirirse conjuntamente.

## 1.3 Objetivos específicos

1. Evaluar la calidad y estructura del conjunto de datos.
2. Preparar una matriz transaccional booleana sin eliminar filas silenciosamente.
3. Identificar productos, pares y tríos frecuentes.
4. Generar reglas con soporte, confianza y lift.
5. Evaluar las reglas con métricas complementarias.
6. Analizar la sensibilidad de los resultados ante diferentes umbrales.
7. Proponer decisiones comerciales y construir una función de recomendación.
8. Exportar automáticamente tablas, gráficos y un resumen ejecutivo.

## 1.4 Criterios iniciales

- Soporte mínimo: **0.08**
- Confianza mínima: **0.50**
- Lift mínimo: **1.00**
- Tamaño máximo del conjunto: **3 productos**

## 1.5 Posibles decisiones de negocio

Los resultados podrán apoyar:

- promociones y paquetes de productos;
- venta cruzada;
- recomendaciones personalizadas;
- organización de productos físicos o digitales;
- campañas dirigidas;
- pruebas A/B antes de implementar cambios;
- seguimiento periódico de patrones de compra.

# 2. Configuración del entorno

In [ ]:
# Instalación automática de dependencias faltantes
import importlib.util
import subprocess
import sys

PAQUETES = {
    "mlxtend": "mlxtend",
    "networkx": "networkx",
    "openpyxl": "openpyxl"
}

for modulo, paquete in PAQUETES.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

print("Dependencias listas.")

In [ ]:
# Importación de librerías
from pathlib import Path
from time import perf_counter
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from IPython.display import display
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda valor: f"{valor:.4f}")

print("Librerías importadas correctamente.")

## 2.1 Conexión con Google Drive y configuración de rutas

El notebook busca automáticamente `basket_analysis.csv` dentro de **Mi unidad**.  
Cuando existan varias copias, puede indicarse la ruta exacta en `RUTA_DATASET_MANUAL`.

In [ ]:
# Montar Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = Path("/content/drive/MyDrive")
    EN_COLAB = True
except ImportError:
    print("No se detectó Google Colab. Se utilizará la carpeta actual.")
    BASE_DRIVE = Path.cwd()
    EN_COLAB = False

# Configuración principal
NOMBRE_DATASET = "basket_analysis.csv"

# Opcional: escriba la ruta completa si conoce la ubicación exacta.
# Ejemplo:
# RUTA_DATASET_MANUAL = "/content/drive/MyDrive/Datasets/basket_analysis.csv"
RUTA_DATASET_MANUAL = ""

CARPETA_SALIDA = BASE_DRIVE / "Reglas_Asociacion_Resultados"
CARPETA_GRAFICOS = CARPETA_SALIDA / "graficos"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
CARPETA_GRAFICOS.mkdir(parents=True, exist_ok=True)

print("Carpeta base:", BASE_DRIVE)
print("Carpeta de resultados:", CARPETA_SALIDA)

In [ ]:
# Localización y carga segura del dataset
if RUTA_DATASET_MANUAL.strip():
    ruta_dataset = Path(RUTA_DATASET_MANUAL.strip())
    if not ruta_dataset.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo en la ruta indicada: {ruta_dataset}"
        )
else:
    coincidencias = sorted(BASE_DRIVE.rglob(NOMBRE_DATASET))

    if not coincidencias:
        raise FileNotFoundError(
            f"No se encontró '{NOMBRE_DATASET}' dentro de {BASE_DRIVE}. "
            "Verifique el nombre o complete RUTA_DATASET_MANUAL."
        )

    ruta_dataset = coincidencias[0]

    if len(coincidencias) > 1:
        print("Se encontraron varias copias:")
        for numero, archivo in enumerate(coincidencias, start=1):
            print(f"  {numero}. {archivo}")
        print("\nSe utilizará la primera. Para elegir otra, complete RUTA_DATASET_MANUAL.")

df_original = pd.read_csv(ruta_dataset)

print("\nDataset cargado correctamente")
print("Ruta:", ruta_dataset)
print("Dimensiones:", df_original.shape)
display(df_original.head())

# 3. Comprensión de los datos

Se revisan dimensiones, tipos, valores faltantes, columnas irrelevantes, duplicados, valores únicos y consistencia del formato transaccional.

In [ ]:
# Resumen estructural
print("Número de transacciones:", df_original.shape[0])
print("Número inicial de columnas:", df_original.shape[1])

resumen_tipos = pd.DataFrame({
    "Tipo": df_original.dtypes.astype(str),
    "No nulos": df_original.notna().sum(),
    "Nulos": df_original.isna().sum(),
    "Valores únicos": df_original.nunique(dropna=False)
})

display(resumen_tipos)
print("Filas completamente duplicadas:", int(df_original.duplicated().sum()))
print(
    "Nota: las transacciones duplicadas no se eliminan, "
    "porque pueden representar compras distintas con el mismo contenido."
)

In [ ]:
# Estandarización de nombres y detección de columnas de índice
df = df_original.copy()
df.columns = [str(columna).strip() for columna in df.columns]

if df.columns.duplicated().any():
    repetidas = df.columns[df.columns.duplicated()].tolist()
    raise ValueError(f"Existen nombres de columnas duplicados: {repetidas}")

columnas_indice = [
    columna for columna in df.columns
    if columna.lower().startswith("unnamed")
    or columna.lower() in {"id", "index", "transaction_id", "transaction id"}
]

print("Columnas candidatas a índice:", columnas_indice)

for columna in df.columns:
    muestra = df[columna].dropna().astype(str).str.strip().unique()[:8]
    print(f"{columna}: {muestra}")

# 4. Preparación de los datos

La preparación conserva todas las transacciones. Los valores válidos se convierten a booleanos y cualquier valor desconocido genera una alerta para evitar modificaciones silenciosas.

In [ ]:
# Eliminación únicamente de columnas identificadoras
datos_sin_indice = df.drop(columns=columnas_indice, errors="ignore").copy()

MAPA_BOOLEANO = {
    "true": True, "false": False,
    "1": True, "0": False,
    "si": True, "sí": True, "no": False,
    "yes": True, "y": True, "n": False,
    "t": True, "f": False
}

def convertir_serie_booleana(serie: pd.Series) -> pd.Series:
    """Convierte una columna de presencia/ausencia a booleano nullable."""
    if pd.api.types.is_bool_dtype(serie):
        return serie.astype("boolean")

    if pd.api.types.is_numeric_dtype(serie):
        convertida = serie.map({1: True, 0: False})
        return convertida.astype("boolean")

    normalizada = serie.astype("string").str.strip().str.lower()
    return normalizada.map(MAPA_BOOLEANO).astype("boolean")

datos_nullable = datos_sin_indice.apply(convertir_serie_booleana)

invalidos = datos_nullable.isna().sum()
invalidos = invalidos[invalidos > 0]

if not invalidos.empty:
    print("Valores no reconocidos por columna:")
    display(invalidos.to_frame("Cantidad"))

    detalle_invalidos = {}
    for columna in invalidos.index:
        mascara = datos_nullable[columna].isna()
        detalle_invalidos[columna] = (
            datos_sin_indice.loc[mascara, columna]
            .astype(str)
            .value_counts()
            .head(10)
            .to_dict()
        )

    print("Ejemplos de valores problemáticos:")
    print(json.dumps(detalle_invalidos, ensure_ascii=False, indent=2))

    raise ValueError(
        "Existen valores que no pueden interpretarse como presencia/ausencia. "
        "Corrija el archivo antes de continuar. No se eliminó ninguna fila."
    )

datos = datos_nullable.astype(bool)

transacciones_vacias = datos.sum(axis=1).eq(0)
cantidad_vacias = int(transacciones_vacias.sum())

print("Dimensiones finales:", datos.shape)
print("Filas conservadas:", len(datos) == len(df_original))
print("Transacciones sin productos:", cantidad_vacias)
print(
    "Las transacciones vacías se conservan para mantener el total original "
    "y el denominador real del soporte."
)
display(datos.head())

In [ ]:
# Reporte de calidad después de la preparación
reporte_calidad = pd.DataFrame({
    "Indicador": [
        "Transacciones originales",
        "Transacciones preparadas",
        "Productos analizados",
        "Valores faltantes",
        "Filas duplicadas",
        "Transacciones vacías",
        "Filas eliminadas"
    ],
    "Valor": [
        len(df_original),
        len(datos),
        datos.shape[1],
        int(datos.isna().sum().sum()),
        int(datos.duplicated().sum()),
        cantidad_vacias,
        len(df_original) - len(datos)
    ]
})

display(reporte_calidad)

# 5. Análisis exploratorio

In [ ]:
# Frecuencia y soporte individual por producto
frecuencia = datos.sum().sort_values(ascending=False)
soporte_individual = frecuencia / len(datos)

resumen_productos = pd.DataFrame({
    "Producto": frecuencia.index,
    "Frecuencia": frecuencia.astype(int).values,
    "Soporte": soporte_individual.values,
    "Porcentaje": (soporte_individual.values * 100)
}).reset_index(drop=True)

display(resumen_productos)

In [ ]:
# Gráfico 1: frecuencia de productos
plt.figure(figsize=(12, 7))
orden = resumen_productos.sort_values("Frecuencia", ascending=True)
plt.barh(orden["Producto"], orden["Frecuencia"])
plt.title("Frecuencia de aparición de los productos")
plt.xlabel("Número de transacciones")
plt.ylabel("Producto")
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "01_frecuencia_productos.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Tamaño de las transacciones
productos_por_transaccion = datos.sum(axis=1)

estadisticas_cesta = productos_por_transaccion.describe().to_frame(
    "Cantidad de productos"
)
display(estadisticas_cesta)

distribucion_cesta = (
    productos_por_transaccion
    .value_counts()
    .sort_index()
    .rename_axis("Productos por transacción")
    .reset_index(name="Transacciones")
)
display(distribucion_cesta)

plt.figure(figsize=(10, 5))
plt.bar(
    distribucion_cesta["Productos por transacción"].astype(str),
    distribucion_cesta["Transacciones"]
)
plt.title("Distribución del tamaño de las transacciones")
plt.xlabel("Número de productos")
plt.ylabel("Número de transacciones")
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "02_tamano_transacciones.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Matriz de coocurrencia de productos
matriz_coocurrencia = datos.astype(int).T.dot(datos.astype(int))
np.fill_diagonal(matriz_coocurrencia.values, 0)

pares_coocurrencia = (
    matriz_coocurrencia
    .where(np.triu(np.ones(matriz_coocurrencia.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
pares_coocurrencia.columns = ["Producto A", "Producto B", "Frecuencia conjunta"]
pares_coocurrencia = pares_coocurrencia.sort_values(
    "Frecuencia conjunta", ascending=False
).reset_index(drop=True)
pares_coocurrencia["Soporte conjunto"] = (
    pares_coocurrencia["Frecuencia conjunta"] / len(datos)
)

print("Pares con mayor coocurrencia:")
display(pares_coocurrencia.head(15))

In [ ]:
# Gráfico 3: mapa de calor de coocurrencias
fig, ax = plt.subplots(figsize=(11, 9))
imagen = ax.imshow(matriz_coocurrencia.values, aspect="auto")
ax.set_xticks(range(len(matriz_coocurrencia.columns)))
ax.set_xticklabels(matriz_coocurrencia.columns, rotation=90)
ax.set_yticks(range(len(matriz_coocurrencia.index)))
ax.set_yticklabels(matriz_coocurrencia.index)
ax.set_title("Matriz de coocurrencia de productos")
fig.colorbar(imagen, ax=ax, label="Número de transacciones conjuntas")
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "03_matriz_coocurrencia.png", dpi=200, bbox_inches="tight")
plt.show()

# 6. Modelado

## 6.1 Diseño de prueba

El algoritmo principal es **Apriori**. Se limita el tamaño de los conjuntos a tres productos para conservar reglas interpretables.

Parámetros iniciales:

- soporte mínimo = 0.08;
- confianza mínima = 0.50;
- lift mínimo = 1.00;
- tamaño máximo = 3.

In [ ]:
# Parámetros del modelo
SOPORTE_MINIMO = 0.08
CONFIANZA_MINIMA = 0.50
LIFT_MINIMO = 1.00
MAX_TAMANO = 3

print("Configuración:")
print(f"Soporte mínimo: {SOPORTE_MINIMO:.2f}")
print(f"Confianza mínima: {CONFIANZA_MINIMA:.2f}")
print(f"Lift mínimo: {LIFT_MINIMO:.2f}")
print(f"Tamaño máximo: {MAX_TAMANO}")

In [ ]:
def preparar_itemsets(itemsets: pd.DataFrame, total_transacciones: int) -> pd.DataFrame:
    """Agrega campos interpretables a los conjuntos frecuentes."""
    salida = itemsets.copy()
    if salida.empty:
        return pd.DataFrame(
            columns=["support", "itemsets", "tamaño", "frecuencia_absoluta", "itemset_texto"]
        )

    salida["tamaño"] = salida["itemsets"].apply(len)
    salida["frecuencia_absoluta"] = (
        salida["support"] * total_transacciones
    ).round().astype(int)
    salida["itemset_texto"] = salida["itemsets"].apply(
        lambda conjunto: ", ".join(sorted(conjunto))
    )
    return salida.sort_values(
        ["support", "tamaño"], ascending=[False, True]
    ).reset_index(drop=True)


def construir_reglas(
    itemsets: pd.DataFrame,
    confianza_minima: float,
    lift_minimo: float,
    total_transacciones: int
) -> pd.DataFrame:
    """Genera y enriquece reglas de asociación."""
    if itemsets.empty or not (itemsets["itemsets"].apply(len) >= 2).any():
        return pd.DataFrame()

    reglas_df = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=confianza_minima
    )

    reglas_df = reglas_df.loc[reglas_df["lift"] >= lift_minimo].copy()

    if reglas_df.empty:
        return reglas_df

    reglas_df["antecedente_texto"] = reglas_df["antecedents"].apply(
        lambda conjunto: ", ".join(sorted(conjunto))
    )
    reglas_df["consecuente_texto"] = reglas_df["consequents"].apply(
        lambda conjunto: ", ".join(sorted(conjunto))
    )
    reglas_df["regla"] = (
        reglas_df["antecedente_texto"] + " → " + reglas_df["consecuente_texto"]
    )
    reglas_df["tamaño_antecedente"] = reglas_df["antecedents"].apply(len)
    reglas_df["tamaño_consecuente"] = reglas_df["consequents"].apply(len)
    reglas_df["frecuencia_conjunta"] = (
        reglas_df["support"] * total_transacciones
    ).round().astype(int)

    denominador_jaccard = (
        reglas_df["antecedent support"]
        + reglas_df["consequent support"]
        - reglas_df["support"]
    )
    reglas_df["jaccard"] = np.where(
        denominador_jaccard > 0,
        reglas_df["support"] / denominador_jaccard,
        np.nan
    )

    reglas_df["kulczynski"] = 0.5 * (
        reglas_df["support"] / reglas_df["antecedent support"]
        + reglas_df["support"] / reglas_df["consequent support"]
    )

    columnas_orden = [
        "antecedents", "consequents",
        "antecedente_texto", "consecuente_texto", "regla",
        "antecedent support", "consequent support",
        "support", "confidence", "lift", "leverage", "conviction",
        "jaccard", "kulczynski",
        "frecuencia_conjunta",
        "tamaño_antecedente", "tamaño_consecuente"
    ]

    columnas_existentes = [
        columna for columna in columnas_orden if columna in reglas_df.columns
    ]

    return (
        reglas_df[columnas_existentes]
        .sort_values(["lift", "confidence", "support"], ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
# Entrenamiento principal con Apriori
inicio_apriori = perf_counter()

itemsets_base = apriori(
    datos,
    min_support=SOPORTE_MINIMO,
    use_colnames=True,
    max_len=MAX_TAMANO,
    low_memory=False
)

tiempo_apriori = perf_counter() - inicio_apriori

itemsets_frecuentes = preparar_itemsets(itemsets_base, len(datos))
reglas = construir_reglas(
    itemsets_base,
    confianza_minima=CONFIANZA_MINIMA,
    lift_minimo=LIFT_MINIMO,
    total_transacciones=len(datos)
)

print(f"Tiempo de Apriori: {tiempo_apriori:.6f} segundos")
print("Conjuntos frecuentes:", len(itemsets_frecuentes))
print("Reglas obtenidas:", len(reglas))

print("\nPrimeros conjuntos frecuentes:")
display(itemsets_frecuentes.head(20))

if reglas.empty:
    print(
        "No se generaron reglas con los umbrales actuales. "
        "Revise el análisis de sensibilidad."
    )
else:
    print("\nPrimeras reglas ordenadas por lift:")
    display(reglas.head(20))

## 6.2 Comparación con FP-Growth

FP-Growth se utiliza como contraste. El objetivo es verificar que, con los mismos parámetros, ambos métodos recuperen los mismos conjuntos frecuentes, aunque su estrategia computacional sea diferente.

In [ ]:
# Comparación entre Apriori y FP-Growth
inicio_fp = perf_counter()

itemsets_fp_base = fpgrowth(
    datos,
    min_support=SOPORTE_MINIMO,
    use_colnames=True,
    max_len=MAX_TAMANO
)

tiempo_fp = perf_counter() - inicio_fp
itemsets_fp = preparar_itemsets(itemsets_fp_base, len(datos))

def diccionario_soportes(itemsets_df):
    return {
        tuple(sorted(fila["itemsets"])): round(float(fila["support"]), 12)
        for _, fila in itemsets_df.iterrows()
    }

coinciden_itemsets = (
    diccionario_soportes(itemsets_base)
    == diccionario_soportes(itemsets_fp_base)
)

comparacion_algoritmos = pd.DataFrame({
    "Algoritmo": ["Apriori", "FP-Growth"],
    "Tiempo_segundos": [tiempo_apriori, tiempo_fp],
    "Itemsets_frecuentes": [len(itemsets_frecuentes), len(itemsets_fp)]
})

display(comparacion_algoritmos)
print("¿Coinciden los conjuntos frecuentes y sus soportes?:", coinciden_itemsets)

# 7. Evaluación de resultados

## 7.1 Métricas

- **Soporte:** proporción de transacciones que contiene antecedente y consecuente.
- **Confianza:** probabilidad del consecuente cuando aparece el antecedente.
- **Lift:** relación observada frente a la esperada bajo independencia.
- **Leverage:** diferencia entre soporte observado y soporte esperado.
- **Conviction:** intensidad de la implicación considerando incumplimientos.
- **Jaccard:** similitud relativa entre antecedente y consecuente.
- **Kulczynski:** promedio de las probabilidades condicionales en ambas direcciones.

Un lift superior a 1 representa asociación positiva, pero no prueba causalidad.

In [ ]:
# Resumen cuantitativo del modelo
distribucion_itemsets = (
    itemsets_frecuentes["tamaño"]
    .value_counts()
    .sort_index()
    .rename_axis("Tamaño del itemset")
    .reset_index(name="Cantidad")
)

resumen_modelo = pd.DataFrame({
    "Indicador": [
        "Transacciones",
        "Productos",
        "Itemsets frecuentes",
        "Reglas",
        "Soporte mínimo",
        "Confianza mínima",
        "Lift mínimo",
        "Mayor lift",
        "Mayor confianza",
        "Mayor soporte de regla"
    ],
    "Valor": [
        len(datos),
        datos.shape[1],
        len(itemsets_frecuentes),
        len(reglas),
        SOPORTE_MINIMO,
        CONFIANZA_MINIMA,
        LIFT_MINIMO,
        reglas["lift"].max() if not reglas.empty else np.nan,
        reglas["confidence"].max() if not reglas.empty else np.nan,
        reglas["support"].max() if not reglas.empty else np.nan
    ]
})

display(resumen_modelo)
display(distribucion_itemsets)

In [ ]:
# Tablas de mejores reglas por diferentes criterios
def mostrar_mejores_reglas(reglas_df, metrica, n=10):
    if reglas_df.empty:
        print("No existen reglas para mostrar.")
        return pd.DataFrame()

    columnas = [
        "regla", "support", "confidence", "lift",
        "leverage", "conviction", "jaccard", "kulczynski",
        "frecuencia_conjunta"
    ]
    columnas = [c for c in columnas if c in reglas_df.columns]

    resultado = (
        reglas_df
        .replace([np.inf, -np.inf], np.nan)
        .nlargest(n, metrica)
        [columnas]
        .copy()
    )
    display(resultado)
    return resultado

print("Top 10 por lift")
top_lift = mostrar_mejores_reglas(reglas, "lift", 10)

print("\nTop 10 por confianza")
top_confianza = mostrar_mejores_reglas(reglas, "confidence", 10)

print("\nTop 10 por soporte")
top_soporte = mostrar_mejores_reglas(reglas, "support", 10)

print("\nTop 10 por leverage")
top_leverage = mostrar_mejores_reglas(reglas, "leverage", 10)

In [ ]:
# Interpretación automática de las reglas principales
if reglas.empty:
    print("No hay reglas disponibles para interpretar.")
else:
    for posicion, (_, fila) in enumerate(reglas.head(10).iterrows(), start=1):
        print(
            f"{posicion}. Si una transacción contiene [{fila['antecedente_texto']}], "
            f"también contiene [{fila['consecuente_texto']}] en "
            f"{fila['confidence']:.2%} de los casos. "
            f"La combinación aparece en {fila['support']:.2%} de todas las transacciones "
            f"y presenta un lift de {fila['lift']:.3f}."
        )

In [ ]:
# Gráfico 4: soporte, confianza y lift
if not reglas.empty:
    tamanos = np.clip(reglas["lift"] * 55, 25, 350)

    plt.figure(figsize=(10, 7))
    dispersion = plt.scatter(
        reglas["support"],
        reglas["confidence"],
        s=tamanos,
        c=reglas["lift"],
        alpha=0.70
    )
    plt.colorbar(dispersion, label="Lift")
    plt.axhline(CONFIANZA_MINIMA, linestyle="--", linewidth=1)
    plt.axvline(SOPORTE_MINIMO, linestyle="--", linewidth=1)
    plt.title("Reglas: soporte, confianza y lift")
    plt.xlabel("Soporte")
    plt.ylabel("Confianza")
    plt.tight_layout()
    plt.savefig(CARPETA_GRAFICOS / "04_soporte_confianza_lift.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("No se genera el gráfico porque no existen reglas.")

In [ ]:
# Gráfico 5: reglas con mayor lift
if not reglas.empty:
    mejores = reglas.nlargest(12, "lift").sort_values("lift")

    plt.figure(figsize=(12, 7))
    plt.barh(mejores["regla"], mejores["lift"])
    plt.axvline(1, linestyle="--", linewidth=1)
    plt.title("Reglas con mayor lift")
    plt.xlabel("Lift")
    plt.ylabel("Regla")
    plt.tight_layout()
    plt.savefig(CARPETA_GRAFICOS / "05_top_reglas_lift.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# Gráfico 6: matriz de lift para reglas de un producto a otro
if not reglas.empty:
    reglas_unarias = reglas.loc[
        (reglas["tamaño_antecedente"] == 1)
        & (reglas["tamaño_consecuente"] == 1)
    ].copy()

    if not reglas_unarias.empty:
        matriz_lift = reglas_unarias.pivot_table(
            index="antecedente_texto",
            columns="consecuente_texto",
            values="lift",
            aggfunc="max"
        )

        fig, ax = plt.subplots(figsize=(11, 9))
        imagen = ax.imshow(matriz_lift.fillna(0).values, aspect="auto")
        ax.set_xticks(range(len(matriz_lift.columns)))
        ax.set_xticklabels(matriz_lift.columns, rotation=90)
        ax.set_yticks(range(len(matriz_lift.index)))
        ax.set_yticklabels(matriz_lift.index)
        ax.set_title("Matriz de lift: reglas de un producto a otro")
        fig.colorbar(imagen, ax=ax, label="Lift")
        plt.tight_layout()
        plt.savefig(CARPETA_GRAFICOS / "06_matriz_lift.png", dpi=200, bbox_inches="tight")
        plt.show()
    else:
        print("No existen reglas unarias para construir la matriz de lift.")

In [ ]:
# Gráfico 7: red de las reglas principales
if not reglas.empty:
    reglas_red = reglas.head(12)
    grafo = nx.DiGraph()

    for _, fila in reglas_red.iterrows():
        origen = fila["antecedente_texto"]
        destino = fila["consecuente_texto"]
        grafo.add_edge(
            origen,
            destino,
            weight=float(fila["lift"]),
            confidence=float(fila["confidence"])
        )

    plt.figure(figsize=(13, 9))
    posiciones = nx.spring_layout(grafo, seed=42, k=1.2)
    pesos = [grafo[u][v]["weight"] for u, v in grafo.edges()]
    anchos = [max(1, (peso - 1) * 4 + 1) for peso in pesos]

    nx.draw_networkx_nodes(grafo, posiciones, node_size=2200)
    nx.draw_networkx_labels(grafo, posiciones, font_size=8)
    nx.draw_networkx_edges(
        grafo,
        posiciones,
        width=anchos,
        arrows=True,
        arrowsize=18,
        connectionstyle="arc3,rad=0.08"
    )

    plt.title("Red de las principales reglas de asociación")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(CARPETA_GRAFICOS / "07_red_reglas.png", dpi=200, bbox_inches="tight")
    plt.show()

## 7.2 Análisis de sensibilidad

Se prueban diferentes combinaciones de soporte y confianza. Esto permite observar el equilibrio entre cantidad de reglas, representatividad y exigencia.

In [ ]:
# Sensibilidad de soporte y confianza
resultados_sensibilidad = []

soportes_prueba = [0.05, 0.08, 0.10, 0.12, 0.15]
confianzas_prueba = [0.40, 0.50, 0.60, 0.70]

for soporte_prueba in soportes_prueba:
    inicio = perf_counter()
    itemsets_prueba = apriori(
        datos,
        min_support=soporte_prueba,
        use_colnames=True,
        max_len=MAX_TAMANO,
        low_memory=False
    )
    tiempo_itemsets = perf_counter() - inicio

    for confianza_prueba in confianzas_prueba:
        reglas_prueba = construir_reglas(
            itemsets_prueba,
            confianza_minima=confianza_prueba,
            lift_minimo=LIFT_MINIMO,
            total_transacciones=len(datos)
        )

        resultados_sensibilidad.append({
            "Soporte mínimo": soporte_prueba,
            "Confianza mínima": confianza_prueba,
            "Lift mínimo": LIFT_MINIMO,
            "Itemsets frecuentes": len(itemsets_prueba),
            "Reglas": len(reglas_prueba),
            "Lift máximo": (
                reglas_prueba["lift"].max()
                if not reglas_prueba.empty else np.nan
            ),
            "Confianza máxima": (
                reglas_prueba["confidence"].max()
                if not reglas_prueba.empty else np.nan
            ),
            "Tiempo itemsets (s)": tiempo_itemsets
        })

sensibilidad = pd.DataFrame(resultados_sensibilidad)
display(sensibilidad)

In [ ]:
# Gráfico 8: número de reglas según umbrales
plt.figure(figsize=(10, 6))

for confianza_prueba in confianzas_prueba:
    subconjunto = sensibilidad.loc[
        sensibilidad["Confianza mínima"] == confianza_prueba
    ]
    plt.plot(
        subconjunto["Soporte mínimo"],
        subconjunto["Reglas"],
        marker="o",
        label=f"Confianza {confianza_prueba:.2f}"
    )

plt.title("Sensibilidad del número de reglas")
plt.xlabel("Soporte mínimo")
plt.ylabel("Número de reglas")
plt.legend()
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "08_sensibilidad_reglas.png", dpi=200, bbox_inches="tight")
plt.show()

# 8. Recomendador basado en reglas

La función siguiente recibe uno o varios productos y busca reglas cuyo antecedente esté contenido en la cesta. Luego ordena los productos sugeridos mediante una puntuación que combina confianza, lift y soporte.

In [ ]:
def recomendar_productos(
    productos_comprados,
    reglas_df=reglas,
    top_n=5
):
    """Recomienda productos a partir de reglas aplicables a una cesta."""
    productos_base = {str(producto).strip() for producto in productos_comprados}

    if not productos_base:
        raise ValueError("Debe indicar al menos un producto.")

    productos_desconocidos = productos_base - set(datos.columns)
    if productos_desconocidos:
        raise ValueError(
            f"Productos no encontrados en el dataset: {sorted(productos_desconocidos)}"
        )

    if reglas_df.empty:
        return pd.DataFrame(
            columns=[
                "Producto recomendado", "Puntuación", "Confianza máxima",
                "Lift máximo", "Soporte máximo", "Reglas de respaldo"
            ]
        )

    candidatos = []

    for _, fila in reglas_df.iterrows():
        antecedente = set(fila["antecedents"])
        consecuente = set(fila["consequents"]) - productos_base

        if antecedente.issubset(productos_base) and consecuente:
            puntuacion = (
                float(fila["confidence"])
                * float(fila["lift"])
                * float(fila["support"])
            )

            for producto in consecuente:
                candidatos.append({
                    "Producto recomendado": producto,
                    "Puntuación": puntuacion,
                    "Confianza": float(fila["confidence"]),
                    "Lift": float(fila["lift"]),
                    "Soporte": float(fila["support"]),
                    "Regla": fila["regla"]
                })

    if not candidatos:
        return pd.DataFrame(
            columns=[
                "Producto recomendado", "Puntuación", "Confianza máxima",
                "Lift máximo", "Soporte máximo", "Reglas de respaldo"
            ]
        )

    candidatos_df = pd.DataFrame(candidatos)

    recomendaciones = (
        candidatos_df
        .groupby("Producto recomendado", as_index=False)
        .agg({
            "Puntuación": "sum",
            "Confianza": "max",
            "Lift": "max",
            "Soporte": "max",
            "Regla": lambda valores: " | ".join(pd.unique(valores)[:3])
        })
        .rename(columns={
            "Confianza": "Confianza máxima",
            "Lift": "Lift máximo",
            "Soporte": "Soporte máximo",
            "Regla": "Reglas de respaldo"
        })
        .sort_values(
            ["Puntuación", "Lift máximo", "Confianza máxima"],
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return recomendaciones


# Ejemplo automático con el producto más frecuente
producto_ejemplo = resumen_productos.iloc[0]["Producto"]
print(f"Ejemplo de recomendaciones para: {producto_ejemplo}")
display(recomendar_productos([producto_ejemplo], top_n=5))

# 9. Interpretación comercial

Las reglas deben transformarse en acciones medibles. Se recomienda priorizar asociaciones con lift mayor que 1, soporte suficiente, confianza alta y sentido comercial.

In [ ]:
# Recomendaciones comerciales automáticas
if reglas.empty:
    print(
        "No se pueden generar recomendaciones comerciales específicas "
        "porque no existen reglas bajo los parámetros actuales."
    )
else:
    print("RECOMENDACIONES PRIORIZADAS\n")

    for numero, (_, fila) in enumerate(reglas.head(5).iterrows(), start=1):
        print(
            f"{numero}. Probar una promoción o recomendación de "
            f"[{fila['consecuente_texto']}] cuando el cliente seleccione "
            f"[{fila['antecedente_texto']}]. "
            f"Confianza: {fila['confidence']:.2%}; "
            f"lift: {fila['lift']:.3f}; "
            f"soporte: {fila['support']:.2%}."
        )

    print("\nAcciones generales:")
    print("- Implementar pruebas A/B antes de aplicar promociones permanentes.")
    print("- No interpretar las reglas como causalidad.")
    print("- Revisar márgenes, inventario y restricciones comerciales.")
    print("- Reentrenar el análisis cuando se incorporen nuevas transacciones.")

# 10. Revisión del proceso

Se verifican los principales controles de calidad antes del despliegue.

In [ ]:
# Lista de verificación
controles = {
    "Dataset cargado": not df_original.empty,
    "Filas conservadas": len(datos) == len(df_original),
    "Sin valores faltantes después de conversión": datos.isna().sum().sum() == 0,
    "Variables booleanas": all(tipo == bool for tipo in datos.dtypes),
    "Itemsets calculados": isinstance(itemsets_frecuentes, pd.DataFrame),
    "Reglas calculadas": isinstance(reglas, pd.DataFrame),
    "Sensibilidad calculada": isinstance(sensibilidad, pd.DataFrame),
    "Carpeta de salida creada": CARPETA_SALIDA.exists()
}

revision_proceso = pd.DataFrame(
    [{"Control": clave, "Cumplido": valor} for clave, valor in controles.items()]
)
display(revision_proceso)

if revision_proceso["Cumplido"].all():
    print("Todos los controles fueron superados.")
else:
    print("Existen controles pendientes. Revise la tabla antes de exportar.")

# 11. Despliegue y exportación

Se generan los cuatro archivos principales solicitados:

1. `itemsets_frecuentes.csv`
2. `reglas_asociacion.csv`
3. `resumen_productos.csv`
4. `analisis_sensibilidad.csv`

Además, se guardan gráficos, comparación de algoritmos, coocurrencias, un libro Excel, un resumen ejecutivo y un archivo ZIP.

In [ ]:
# Preparación de versiones exportables
itemsets_exportar = itemsets_frecuentes.drop(columns=["itemsets"], errors="ignore").copy()

reglas_exportar = reglas.drop(
    columns=["antecedents", "consequents"],
    errors="ignore"
).copy()

# Rutas principales
ruta_itemsets = CARPETA_SALIDA / "itemsets_frecuentes.csv"
ruta_reglas = CARPETA_SALIDA / "reglas_asociacion.csv"
ruta_resumen = CARPETA_SALIDA / "resumen_productos.csv"
ruta_sensibilidad = CARPETA_SALIDA / "analisis_sensibilidad.csv"
ruta_coocurrencias = CARPETA_SALIDA / "coocurrencias_pares.csv"
ruta_comparacion = CARPETA_SALIDA / "comparacion_algoritmos.csv"
ruta_revision = CARPETA_SALIDA / "revision_proceso.csv"
ruta_excel = CARPETA_SALIDA / "reporte_reglas_asociacion.xlsx"
ruta_resumen_txt = CARPETA_SALIDA / "resumen_ejecutivo.txt"

# Exportación CSV
itemsets_exportar.to_csv(ruta_itemsets, index=False, encoding="utf-8-sig")
reglas_exportar.to_csv(ruta_reglas, index=False, encoding="utf-8-sig")
resumen_productos.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")
sensibilidad.to_csv(ruta_sensibilidad, index=False, encoding="utf-8-sig")
pares_coocurrencia.to_csv(ruta_coocurrencias, index=False, encoding="utf-8-sig")
comparacion_algoritmos.to_csv(ruta_comparacion, index=False, encoding="utf-8-sig")
revision_proceso.to_csv(ruta_revision, index=False, encoding="utf-8-sig")

# Libro Excel consolidado
with pd.ExcelWriter(ruta_excel, engine="openpyxl") as escritor:
    resumen_productos.to_excel(escritor, sheet_name="Productos", index=False)
    itemsets_exportar.to_excel(escritor, sheet_name="Itemsets", index=False)
    reglas_exportar.to_excel(escritor, sheet_name="Reglas", index=False)
    sensibilidad.to_excel(escritor, sheet_name="Sensibilidad", index=False)
    comparacion_algoritmos.to_excel(escritor, sheet_name="Algoritmos", index=False)
    pares_coocurrencia.to_excel(escritor, sheet_name="Coocurrencias", index=False)
    revision_proceso.to_excel(escritor, sheet_name="Validacion", index=False)

# Resumen ejecutivo
producto_mas_frecuente = resumen_productos.iloc[0]["Producto"]
soporte_producto_principal = resumen_productos.iloc[0]["Soporte"]

lineas_resumen = [
    "RESUMEN EJECUTIVO - REGLAS DE ASOCIACIÓN",
    "=" * 48,
    f"Dataset: {ruta_dataset}",
    f"Transacciones analizadas: {len(datos)}",
    f"Productos analizados: {datos.shape[1]}",
    f"Producto más frecuente: {producto_mas_frecuente}",
    f"Soporte del producto más frecuente: {soporte_producto_principal:.2%}",
    f"Itemsets frecuentes: {len(itemsets_frecuentes)}",
    f"Reglas obtenidas: {len(reglas)}",
    f"Soporte mínimo: {SOPORTE_MINIMO:.2f}",
    f"Confianza mínima: {CONFIANZA_MINIMA:.2f}",
    f"Lift mínimo: {LIFT_MINIMO:.2f}",
    f"Coincidencia Apriori/FP-Growth: {coinciden_itemsets}",
]

if not reglas.empty:
    mejor = reglas.iloc[0]
    lineas_resumen.extend([
        "",
        "Regla principal por lift:",
        mejor["regla"],
        f"Soporte: {mejor['support']:.2%}",
        f"Confianza: {mejor['confidence']:.2%}",
        f"Lift: {mejor['lift']:.4f}",
        "",
        "La regla representa asociación estadística y no causalidad."
    ])

ruta_resumen_txt.write_text(
    "\n".join(lineas_resumen),
    encoding="utf-8"
)

# Comprimir resultados
ruta_zip_base = CARPETA_SALIDA.parent / "Reglas_Asociacion_Resultados"
ruta_zip = shutil.make_archive(
    str(ruta_zip_base),
    "zip",
    root_dir=CARPETA_SALIDA
)

print("Exportación completada:")
for ruta in [
    ruta_itemsets, ruta_reglas, ruta_resumen, ruta_sensibilidad,
    ruta_coocurrencias, ruta_comparacion, ruta_revision,
    ruta_excel, ruta_resumen_txt, Path(ruta_zip)
]:
    print("-", ruta)

# 12. Conclusiones

La celda siguiente genera conclusiones usando los resultados reales obtenidos durante la ejecución.

In [ ]:
# Conclusiones automáticas
print("CONCLUSIONES\n")

print(
    f"1. Se analizaron {len(datos)} transacciones y {datos.shape[1]} productos, "
    "conservando todas las filas del dataset."
)

print(
    f"2. El producto con mayor presencia fue "
    f"{resumen_productos.iloc[0]['Producto']}, con un soporte de "
    f"{resumen_productos.iloc[0]['Soporte']:.2%}."
)

print(
    f"3. Con soporte mínimo {SOPORTE_MINIMO:.2f} y tamaño máximo "
    f"{MAX_TAMANO}, Apriori identificó {len(itemsets_frecuentes)} "
    "conjuntos frecuentes."
)

print(
    f"4. Con confianza mínima {CONFIANZA_MINIMA:.2f} y lift mínimo "
    f"{LIFT_MINIMO:.2f}, se obtuvieron {len(reglas)} reglas."
)

if not reglas.empty:
    mejor = reglas.iloc[0]
    print(
        f"5. La regla con mayor lift fue [{mejor['regla']}], "
        f"con soporte {mejor['support']:.2%}, confianza "
        f"{mejor['confidence']:.2%} y lift {mejor['lift']:.3f}."
    )
else:
    print(
        "5. No se obtuvieron reglas bajo los umbrales iniciales; "
        "el análisis de sensibilidad permite seleccionar otros valores."
    )

print(
    f"6. Apriori y FP-Growth "
    f"{'produjeron los mismos itemsets frecuentes' if coinciden_itemsets else 'presentaron diferencias que deben revisarse'}."
)

print(
    "7. Los resultados pueden apoyar promociones, venta cruzada, "
    "organización de productos y recomendaciones, siempre mediante "
    "validación comercial y pruebas controladas."
)

# 13. Próximos pasos

1. Incorporar fecha, sucursal, cliente o categoría cuando esas variables estén disponibles.
2. Comparar reglas por periodos y segmentos.
3. Analizar rentabilidad y disponibilidad de inventario.
4. Implementar pruebas A/B para evaluar promociones.
5. Reentrenar el modelo periódicamente.
6. Integrar el recomendador en una aplicación o tablero.